# Recs 006: Evaluate review-style queries (raw vs structured)

## Key Goal

Run a 4-way ablation (raw/structured query x raw/structured index) to locate where structure helps most.

## Decision It Supports

Whether structure should be applied on query side, index side, both, or neither.

## Primary Metrics

Proxy ranking metrics across four arms: Hit@K, Recall@K, MAP@K, NDCG@K, MRR.

## Framing

This notebook answers **where structured preference text helps**: on the query only, the game index only, both, or neither—via a **4-way** design (raw vs structured × query vs index). In our reported runs, **raw query embeddings with raw game embeddings** performed best overall; structured text stays in the story as an explicit comparison axis.

**Compared to recs_004:** recs_004 benchmarks **many query construction variants** against one index setup; recs_006 holds the lens on **paired raw/structured representations** across query and catalog.

## Optional Exploration Note

Hypothesis: structured rewrites may not show gains with Universal Sentence Encoder (USE) because the embedding model may not fully leverage richer instruction-style structure. A useful follow-up is to rerun this 4-way comparison with a stronger LLM embedding model to test whether structured query/index text starts to outperform raw under higher-capacity embeddings.


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import pandas as pd

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from cwd={here}")

REPO_ROOT = _repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

EVAL_PATH = REPO_ROOT / "artifacts" / "recs" / "eval_queries_review_style.jsonl"
OUT_DIR = REPO_ROOT / "artifacts" / "recs"
OUT_ROWS_JSONL = OUT_DIR / "eval_review_style_ab_rows.jsonl"
OUT_SUMMARY_CSV = OUT_DIR / "eval_review_style_ab_summary.csv"

print("Repo root:", REPO_ROOT)
print("Eval file:", EVAL_PATH)

Repo root: /home/ryanr/workspace/steam_recommendations
Eval file: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_queries_review_style.jsonl


In [2]:
from steam_review_ml.recommender.retrieve import ContentRetriever

retriever = ContentRetriever()
print("Loaded index rows:", len(retriever.index_frame))

Loaded index rows: 315


In [3]:
def _load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

queries = _load_jsonl(EVAL_PATH)
print("Loaded queries:", len(queries))
queries[0]

Loaded queries: 31


{'id': 'r001',
 'review_draft': 'Combat feels amazing when parries click, and boss fights are memorable. The story is kind of cryptic and sometimes I had no clue where to go next.',
 'expected_themes': ['challenging melee combat',
  'boss-focused action',
  'dark fantasy tone'],
 'avoid_themes': ['easy casual gameplay'],
 'notes': 'Should map from opinionated review text to soulslike-style preferences.'}

In [4]:
#show more of the pandas dataframe. increase settings 
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.float_format', '{:.5f}'.format)


In [5]:
TOP_K = 10
rows_out: list[dict] = []

for qrow in queries:
    qid = qrow["id"]
    draft = qrow["review_draft"]
    expected = qrow.get("expected_themes", [])
    avoid = qrow.get("avoid_themes", [])

    raw_hits = retriever.top_k(draft, k=TOP_K, structured=False)
    struct_hits = retriever.top_k(draft, k=TOP_K, structured=True)

    raw_names = raw_hits["app_name"].tolist()
    struct_names = struct_hits["app_name"].tolist()

    rows_out.append(
        {
            "id": qid,
            "review_draft": draft,
            "expected_themes": expected,
            "avoid_themes": avoid,
            "raw_top10": raw_names,
            "structured_top10": struct_names,
            "raw_top1": raw_names[0] if raw_names else None,
            "structured_top1": struct_names[0] if struct_names else None,
            "overlap_count_top10": len(set(raw_names).intersection(struct_names)),
            "winner_manual": "",
            "notes_manual": "",
        }
    )

ab_df = pd.DataFrame(rows_out)

2026-04-21 18:15:51.283936: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776809751.307696  270901 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776809751.314360  270901 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776809751.332416  270901 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776809751.332486  270901 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776809751.332488  270901 computation_placer.cc:177] computation placer alr

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

with OUT_ROWS_JSONL.open("w", encoding="utf-8") as f:
    for row in rows_out:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

summary_cols = [
    "id",
    "raw_top1",
    "structured_top1",
    "overlap_count_top10",
    "winner_manual",
    "notes_manual",
]
ab_df[summary_cols].to_csv(OUT_SUMMARY_CSV, index=False)

print("Wrote:")
print("-", OUT_ROWS_JSONL)
print("-", OUT_SUMMARY_CSV)

Wrote:
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_review_style_ab_rows.jsonl
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_review_style_ab_summary.csv


In [7]:
ab_df.head(50)

,id,review_draft,expected_themes,avoid_themes,raw_top10,structured_top10,raw_top1,structured_top1,overlap_count_top10,winner_manual,notes_manual
0,r001,"Combat feels amazing when parries click, and boss fights are memorable. The story is kind of cryptic and sometimes I had no clue where to go next.","[challenging melee combat, boss-focused action, dark fantasy tone]",[easy casual gameplay],"[Titan Souls, Momodora: Reverie Under the Moonlight, Darksiders III, Sekiro™: Shadows Die Twice, Salt and Sanctuary, Hollow Knight, Iconoclasts, Shadow Complex Remastered, Tales of Berseria, Dungreed]","[Darksiders III, Titan Souls, Vampyr, Momodora: Reverie Under the Moonlight, Iconoclasts, Hollow Knight, DRAGON QUEST HEROES™ II, Mutant Year Zero: Road to Eden, Ni no Kuni™ II: Revenant Kingdom, Salt and Sanctuary]",Titan Souls,Darksiders III,6,,
1,r002,"Loved the farming loop and decorating my house. Days were relaxing and I could just chill, but the mines got repetitive after a while.","[cozy progression, farming/life sim, relaxing pace]",[combat-heavy grind],"[Farm Together, Townscaper, Stardew Valley, A Short Hike, House Flipper, Farm Manager 2018, Staxel, The Sims(TM) 3, My Time At Portia, Slime Rancher]","[Farm Together, Townscaper, House Flipper, A Short Hike, Farm Manager 2018, Staxel, Stardew Valley, Kingdom Two Crowns, My Time At Portia, FAR: Lone Sails]",Farm Together,Farm Together,8,,
2,r003,"The squad tactics are the best part: flanking, cover, and risky plays. Missing a 90 percent shot still drives me insane though.","[turn-based tactics, squad strategy, positioning]",[real-time twitch gameplay],"[Sniper Elite 4, Tom Clancy's Rainbow Six Siege, Steel Division: Normandy 44, XCOM 2, Football Manager 2019, Takedown: Red Sabre, Due Process, ULTRAKILL, Dead by Daylight, Day of Infamy]","[XCOM 2, Mutant Year Zero: Road to Eden, Sniper Elite 4, Tom Clancy's Rainbow Six Siege, Warhammer 40,000: Mechanicus, Football Manager 2019, Takedown: Red Sabre, Steel Division: Normandy 44, Bomber Crew, Cold Waters]",Sniper Elite 4,XCOM 2,6,,
3,r004,"Characters were great and I cared about their arcs. Side quests actually had consequences, unlike most open-world filler.","[narrative RPG, companion writing, choices/consequences]",[checklist open world],"[Tales of Berseria, Assassin's Creed Odyssey, DRAGON QUEST HEROES™ II, Fairy Fencer F Advent Dark Force, The Legend of Heroes: Trails of Cold Steel II, Pillars of Eternity II: Deadfire, Ni no Kuni™ II: Revenant Kingdom, Torment: Tides of Numenera, The Walking Dead, Sword Art Online: Fatal Bullet]","[Tales of Berseria, Pillars of Eternity II: Deadfire, Torment: Tides of Numenera, Vampyr, DRAGON QUEST HEROES™ II, Ni no Kuni™ II: Revenant Kingdom, Mutant Year Zero: Road to Eden, Fairy Fencer F Advent Dark Force, The Legend of Heroes: Trails of Cold Steel II, Iconoclasts]",Tales of Berseria,Tales of Berseria,7,,
4,r005,"Finished it in one weekend. Short, stylish, and atmospheric. I appreciated that it did not overstay its welcome.","[short indie, atmospheric experience, concise design]",[very long runtime],"[FAR: Lone Sails, Grimm's Hollow, A Short Hike, Helltaker, There Is No Game: Wrong Dimension, Gunpoint, The Room, Little Nightmares, We Were Here Too, The Room Two]","[FAR: Lone Sails, Little Nightmares, Grimm's Hollow, Mutant Year Zero: Road to Eden, Detention, The Room Two, The Room, Iconoclasts, A Short Hike, There Is No Game: Wrong Dimension]",FAR: Lone Sails,FAR: Lone Sails,7,,
5,r006,City planning and logistics are deep and satisfying. Traffic management is brutal but in a good way.,"[city builder, management sim, optimization/logistics]",[light arcade gameplay],"[Cities: Skylines, Euro Truck Simulator 2, Railway Empire, Banished, Urban Empire, American Truck Simulator, Frostpunk, Rise of Industry, Steel Division: Normandy 44, Ancestors Legacy]","[Urban Empire, Euro Truck Simulator 2, Railway Empire, Surviving Mars, American Truck Simulator, Frostpunk, Banished, Cities: Skylines, Mutant Year Ze

In [8]:
# Optional quick filter to inspect Civ-like example.
ab_df[ab_df["id"] == "r031"][["id", "review_draft", "raw_top10", "structured_top10"]]

,id,review_draft,raw_top10,structured_top10
30,r031,"I keep coming back for the one-more-turn feeling. Expanding cities, balancing science and culture, and deciding when to go to war is incredibly satisfying. Late-game turns can drag, but I still love the empire planning.","[Stellaris, Sid Meier's Civilization V, Sid Meier's Civilization VI, Europa Universalis IV, Hearts of Iron IV, Total War Saga: Thrones of Britannia, STAR WARS™ Empire at War: Gold Pack, Urban Empire, Surviving Mars, Into the Breach]","[Urban Empire, Sid Meier's Civilization VI, Stellaris, Sid Meier's Civilization V, Europa Universalis IV, Total War Saga: Thrones of Britannia, Hearts of Iron IV, Surviving Mars, Warhammer 40,000: Mechanicus, Steel Division: Normandy 44]"


## Structured game index input for comparison

This notebook **loads** the structured game index artifacts built in `notebooks/models/game_embeddings/recs_005_game_embeddings_structured.ipynb`, then runs the 4-way comparison and same-user proxy metrics.


In [9]:
import numpy as np

STRUCTURED_EVAL_INDEX_NPZ = OUT_DIR / "game_profile_embeddings_structured_eval.npz"
STRUCTURED_EVAL_INDEX_PARQUET = OUT_DIR / "game_profile_embedding_index_structured_eval.parquet"

if not STRUCTURED_EVAL_INDEX_NPZ.is_file() or not STRUCTURED_EVAL_INDEX_PARQUET.is_file():
    raise FileNotFoundError(
        "Missing structured index artifacts. Run game_embeddings/recs_005_game_embeddings_structured.ipynb first."
    )

with np.load(STRUCTURED_EVAL_INDEX_NPZ) as z:
    X_struct = np.asarray(z["embeddings"], dtype=np.float32)
    app_ids_struct = np.asarray(z["app_id"], dtype=np.int64)
idx_struct = pd.read_parquet(STRUCTURED_EVAL_INDEX_PARQUET)

print("Loaded structured matrix:", X_struct.shape)
print("Loaded structured index rows:", len(idx_struct))


Loaded structured matrix: (315, 512)
Loaded structured index rows: 315


In [10]:
# Align both indexes to common app_ids so comparisons are apples-to-apples.
raw_df = retriever.index_frame[["app_id", "app_name"]].copy()
raw_ids = retriever.app_ids
raw_X = retriever.embedding_matrix

raw_row_by_id = {int(a): i for i, a in enumerate(raw_ids.tolist())}
struct_row_by_id = {int(a): i for i, a in enumerate(app_ids_struct.tolist())}

common_ids = sorted(set(raw_row_by_id).intersection(struct_row_by_id))
print("Common app_id count:", len(common_ids), "of raw", len(raw_ids), "and structured", len(app_ids_struct))

raw_rows = np.asarray([raw_row_by_id[a] for a in common_ids], dtype=np.int64)
struct_rows = np.asarray([struct_row_by_id[a] for a in common_ids], dtype=np.int64)

X_raw_common = raw_X[raw_rows]
X_struct_common = X_struct[struct_rows]
app_ids_common = np.asarray(common_ids, dtype=np.int64)
idx_common = raw_df.set_index("app_id").loc[common_ids].reset_index(drop=False)

print("Raw common matrix:", X_raw_common.shape)
print("Structured common matrix:", X_struct_common.shape)

Common app_id count: 315 of raw 315 and structured 315
Raw common matrix: (315, 512)
Structured common matrix: (315, 512)


In [11]:
def _query_to_text_for_embedding(query_text: str, structured_query: bool) -> str:
    from steam_review_ml.recommender.preferences import (
        build_embedding_input,
        extract_preferences,
    )

    raw = (query_text or "").strip()
    if not structured_query:
        return raw
    prefs = extract_preferences(raw)
    return build_embedding_input(prefs, raw)


def _top_k_from_matrix(
    retriever: ContentRetriever,
    query_text: str,
    *,
    k: int,
    matrix: np.ndarray,
    app_ids: np.ndarray,
    idx_df: pd.DataFrame,
    structured_query: bool,
    exclude_app_ids: set[int] | None = None,
) -> pd.DataFrame:
    to_embed = _query_to_text_for_embedding(query_text, structured_query=structured_query)
    qv = retriever.embed_text(to_embed)
    sims = (matrix @ qv).astype(np.float32)

    if exclude_app_ids:
        app_row = {int(a): i for i, a in enumerate(app_ids.tolist())}
        for aid in exclude_app_ids:
            row = app_row.get(int(aid))
            if row is not None:
                sims[row] = -np.inf

    kk = min(int(k), len(sims))
    row_idx = np.argpartition(-sims, kk - 1)[:kk]
    order = np.argsort(-sims[row_idx])
    row_idx = row_idx[order]

    out = idx_df.iloc[row_idx].copy().reset_index(drop=True)
    out["score"] = sims[row_idx]
    out["app_id"] = app_ids[row_idx]
    return out


def _positive_set_for_query(qrow: dict, idx_df: pd.DataFrame) -> set[int]:
    """Optional labels:
    - relevant_app_ids: [int]
    - relevant_app_names: [str] (exact, case-insensitive)
    """
    pos: set[int] = set()

    for a in qrow.get("relevant_app_ids", []) or []:
        try:
            pos.add(int(a))
        except Exception:
            pass

    names = qrow.get("relevant_app_names", []) or []
    if names:
        name_map = {
            str(name).strip().lower(): int(app_id)
            for app_id, name in idx_df[["app_id", "app_name"]].itertuples(index=False)
        }
        for n in names:
            key = str(n).strip().lower()
            if key in name_map:
                pos.add(name_map[key])

    return pos

In [12]:
import math

def precision_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    if not positives:
        return float("nan")
    if k <= 0:
        return 0.0
    top = ranked_app_ids[:k]
    if len(top) == 0:
        return 0.0
    hits = sum(int(int(a) in positives) for a in top)
    return hits / float(len(top))


def recall_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    if not positives:
        return float("nan")
    top = set(int(a) for a in ranked_app_ids[:k])
    return len(top & positives) / len(positives)


def average_precision_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    """Binary relevance: average precision truncated at *k*, normalized by |positives|."""
    if not positives:
        return float("nan")
    hits = 0
    prec_sum = 0.0
    for rank, a in enumerate(ranked_app_ids[:k].tolist(), start=1):
        if int(a) in positives:
            hits += 1
            prec_sum += hits / rank
    return prec_sum / len(positives)


def ndcg_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    """Binary NDCG@k: relevance 1 for positives in the top-*k* list."""
    if not positives:
        return float("nan")
    gains = [1.0 if int(a) in positives else 0.0 for a in ranked_app_ids[:k]]

    def dcg(g: list[float]) -> float:
        return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(g))

    ideal_len = min(len(positives), k)
    ideal_gains = [1.0] * ideal_len + [0.0] * max(0, k - ideal_len)
    idcg = dcg(ideal_gains)
    if idcg <= 1e-12:
        return 0.0
    return dcg(gains) / idcg


def mrr(ranked_app_ids: np.ndarray, positives: set[int]) -> float:
    if not positives:
        return float("nan")
    for i, a in enumerate(ranked_app_ids, start=1):
        if int(a) in positives:
            return 1.0 / float(i)
    return 0.0



In [13]:
KS = (5, 10)

In [14]:
KS_4WAY = (5, 10)

arms = {
    "rawQ_rawIdx": dict(matrix=X_raw_common, structured_query=False),
    "structQ_rawIdx": dict(matrix=X_raw_common, structured_query=True),
    "rawQ_structIdx": dict(matrix=X_struct_common, structured_query=False),
    "structQ_structIdx": dict(matrix=X_struct_common, structured_query=True),
}

rows_4way: list[dict] = []

for qrow in queries:
    qid = qrow["id"]
    draft = qrow["review_draft"]
    pos = _positive_set_for_query(qrow, idx_common)

    row = {
        "id": qid,
        "review_draft": draft,
        "expected_themes": qrow.get("expected_themes", []),
        "avoid_themes": qrow.get("avoid_themes", []),
        "n_positives": len(pos),
    }

    for arm_name, cfg in arms.items():
        hits = _top_k_from_matrix(
            retriever,
            draft,
            k=TOP_K,
            matrix=cfg["matrix"],
            app_ids=app_ids_common,
            idx_df=idx_common,
            structured_query=cfg["structured_query"],
        )
        app_ids_ranked = hits["app_id"].to_numpy(dtype=np.int64)
        names_ranked = hits["app_name"].tolist()

        row[f"{arm_name}_top1"] = names_ranked[0] if names_ranked else None
        row[f"{arm_name}_top10"] = names_ranked

        if len(pos) > 0:
            for k in KS_4WAY:
                row[f"{arm_name}_precision@{k}"] = precision_at_k(app_ids_ranked, pos, k)
                row[f"{arm_name}_recall@{k}"] = recall_at_k(app_ids_ranked, pos, k)
                row[f"{arm_name}_map@{k}"] = average_precision_at_k(app_ids_ranked, pos, k)
                row[f"{arm_name}_ndcg@{k}"] = ndcg_at_k(app_ids_ranked, pos, k)
            row[f"{arm_name}_mrr"] = mrr(app_ids_ranked, pos)
        else:
            for k in KS_4WAY:
                row[f"{arm_name}_precision@{k}"] = np.nan
                row[f"{arm_name}_recall@{k}"] = np.nan
                row[f"{arm_name}_map@{k}"] = np.nan
                row[f"{arm_name}_ndcg@{k}"] = np.nan
            row[f"{arm_name}_mrr"] = np.nan

    rows_4way.append(row)

ab4_df = pd.DataFrame(rows_4way)
ab4_df.head(5)

,id,review_draft,expected_themes,avoid_themes,n_positives,rawQ_rawIdx_top1,rawQ_rawIdx_top10,rawQ_rawIdx_precision@5,rawQ_rawIdx_recall@5,rawQ_rawIdx_map@5,rawQ_rawIdx_ndcg@5,rawQ_rawIdx_precision@10,rawQ_rawIdx_recall@10,rawQ_rawIdx_map@10,rawQ_rawIdx_ndcg@10,rawQ_rawIdx_mrr,structQ_rawIdx_top1,structQ_rawIdx_top10,structQ_rawIdx_precision@5,structQ_rawIdx_recall@5,structQ_rawIdx_map@5,structQ_rawIdx_ndcg@5,structQ_rawIdx_precision@10,structQ_rawIdx_recall@10,structQ_rawIdx_map@10,structQ_rawIdx_ndcg@10,structQ_rawIdx_mrr,rawQ_structIdx_top1,rawQ_structIdx_top10,rawQ_structIdx_precision@5,rawQ_structIdx_recall@5,rawQ_structIdx_map@5,rawQ_structIdx_ndcg@5,rawQ_structIdx_precision@10,rawQ_structIdx_recall@10,rawQ_structIdx_map@10,rawQ_structIdx_ndcg@10,rawQ_structIdx_mrr,structQ_structIdx_top1,structQ_structIdx_top10,structQ_structIdx_precision@5,structQ_structIdx_recall@5,structQ_structIdx_map@5,structQ_structIdx_ndcg@5,structQ_structIdx_precision@10,structQ_structIdx_recall@10,structQ_structIdx_map@10,structQ_structIdx_ndcg@10,structQ_structIdx_mrr
0,r001,"Combat feels amazing when parries click, and boss fights are memorable. The story is kind of cryptic and sometimes I had no clue where to go next.","[challenging melee combat, boss-focused action, dark fantasy tone]",[easy casual gameplay],0,Titan Souls,"[Titan Souls, Momodora: Reverie Under the Moonlight, Darksiders III, Sekiro™: Shadows Die Twice, Salt and Sanctuary, Hollow Knight, Iconoclasts, Shadow Complex Remastered, Tales of Berseria, Dungreed]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Darksiders III,"[Darksiders III, Titan Souls, Vampyr, Momodora: Reverie Under the Moonlight, Iconoclasts, Hollow Knight, DRAGON QUEST HEROES™ II, Mutant Year Zero: Road to Eden, Ni no Kuni™ II: Revenant Kingdom, Salt and Sanctuary]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Darksiders III,"[Darksiders III, Tales of Berseria, Momodora: Reverie Under the Moonlight, Dead Cells, Mutant Year Zero: Road to Eden, Titan Souls, Salt and Sanctuary, Vampyr, Guacamelee! Super Turbo Championship Edition, Styx: Shards of Darkness]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Darksiders III,"[Darksiders III, Momodora: Reverie Under the Moonlight, Titan Souls, Dead Cells, Tales of Berseria, Vampyr, Salt and Sanctuary, X-Blades, Guacamelee! Super Turbo Championship Edition, Shadow Complex Remastered]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,r002,"Loved the farming loop and decorating my house. Days were relaxing and I could just chill, but the mines got repetitive after a while.","[cozy progression, farming/life sim, relaxing pace]",[combat-heavy grind],0,Farm Together,"[Farm Together, Townscaper, Stardew Valley, A Short Hike, House Flipper, Farm Manager 2018, Staxel, The Sims(TM) 3, My Time At Portia, Slime Rancher]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Farm Together,"[Farm Together, Townscaper, House Flipper, A Short Hike, Farm Manager 2018, Staxel, Stardew Valley, Kingdom Two Crowns, My Time At Portia, FAR: Lone Sails]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Farm Together,"[Farm Together, Stardew Valley, A Short Hike, Staxel, My Time At Portia, Factorio, Slime Rancher, Farm Manager 2018, House Flipper, Fishing: Barents Sea]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Farm Together,"[Farm Together, A Short Hike, Townscaper, Stardew Valley, House Flipper, FAR: Lone Sails, Staxel, Farm Manager 2018, Slime Rancher, Frostpunk]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,r003,"The squad tactics are the best part: flanking, cover, and risky plays. Missing a 90 percent shot still drives me insane though.","[turn-based tactics, squad strategy, positioning]",[real-time twitch gameplay],0,Sniper Elite 4,"[Sniper Elite 4, Tom Clancy's Rainbow Six Siege, Steel Division: Normandy 44, XCOM 2, Football Manager 2019, Takedown: Red Sabre, Due Process, ULTRAKILL, Dead by Daylight, Day of Infamy]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,XCOM 2,"[XCOM 2, Mutant Year Zero: Road to Eden, Sniper Elite 4, Tom Clancy's Rainbow Six Siege, Warhammer 40,000: Mechanicu

In [15]:
metric_cols = [
    c
    for c in ab4_df.columns
    if any(tag in c for tag in ["precision@", "recall@", "map@", "ndcg@", "mrr"])
]

labeled_df = ab4_df[ab4_df["n_positives"] > 0].copy()
print("Queries with relevance labels:", len(labeled_df), "out of", len(ab4_df))

if len(labeled_df) > 0:
    summary = labeled_df[metric_cols].mean(numeric_only=True).sort_index()
    display(summary.to_frame("mean"))
else:
    print(
        "No relevance labels found yet. Add `relevant_app_ids` or `relevant_app_names` to eval JSONL rows "
        "to compute precision/recall/MAP/NDCG/MRR."
    )

AB4_ROWS_JSONL = OUT_DIR / "eval_review_style_4way_rows.jsonl"
AB4_SUMMARY_CSV = OUT_DIR / "eval_review_style_4way_summary.csv"

with AB4_ROWS_JSONL.open("w", encoding="utf-8") as f:
    for row in rows_4way:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

ab4_df.to_csv(AB4_SUMMARY_CSV, index=False)
print("Wrote:")
print("-", AB4_ROWS_JSONL)
print("-", AB4_SUMMARY_CSV)

Queries with relevance labels: 0 out of 31
No relevance labels found yet. Add `relevant_app_ids` or `relevant_app_names` to eval JSONL rows to compute precision/recall/MAP/NDCG/MRR.
Wrote:
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_review_style_4way_rows.jsonl
- /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_review_style_4way_summary.csv


In [16]:
# Quick sanity slice for your Civilization-style query.
ab4_df[ab4_df["id"] == "r031"][[
    "id",
    "rawQ_rawIdx_top1",
    "structQ_rawIdx_top1",
    "rawQ_structIdx_top1",
    "structQ_structIdx_top1",
]]

,id,rawQ_rawIdx_top1,structQ_rawIdx_top1,rawQ_structIdx_top1,structQ_structIdx_top1
30,r031,Stellaris,Urban Empire,Urban Empire,Europa Universalis IV


## Same-user proxy metrics (recs_004-style) for 4-way comparison

`ab4_df` is great for qualitative inspection, but aggregate ranking metrics require known positives.

This block computes `hit@k`, `recall@k`, `MAP@k`, `NDCG@k`, and `MRR` using a same-user proxy built from the validation split:

- per user: pick one positive review as query text
- treat that user's other positive games as relevant targets
- evaluate each of the 4 retrieval arms on the shared candidate universe



In [17]:
import os

from steam_review_ml.constants import PROJECT_RANDOM_SEED

PROCESSED = REPO_ROOT / "data" / "processed"
VAL_PARQUET = PROCESSED / "steam_reviews_cleaned_english_val_norm.parquet"
TRAIN_PARQUET = PROCESSED / "steam_reviews_cleaned_english_train_norm.parquet"
TEST_PARQUET = PROCESSED / "steam_reviews_cleaned_english_test_norm.parquet"

_split = os.environ.get("RECS004_EVAL_SPLIT", "val").strip().lower()
if _split == "test":
    if not TEST_PARQUET.is_file():
        raise FileNotFoundError(f"RECS004_EVAL_SPLIT=test but missing {TEST_PARQUET}")
    EVAL_PARQUET = TEST_PARQUET
    EVAL_SPLIT_NAME = "test"
elif _split == "train":
    if not TRAIN_PARQUET.is_file():
        raise FileNotFoundError(f"RECS004_EVAL_SPLIT=train but missing {TRAIN_PARQUET}")
    EVAL_PARQUET = TRAIN_PARQUET
    EVAL_SPLIT_NAME = "train"
else:
    EVAL_PARQUET = VAL_PARQUET if VAL_PARQUET.is_file() else TRAIN_PARQUET
    EVAL_SPLIT_NAME = "val" if EVAL_PARQUET == VAL_PARQUET else "train"

if not EVAL_PARQUET.is_file():
    raise FileNotFoundError(f"Missing eval parquet: {EVAL_PARQUET}")

USER_COL_CANDIDATES = ["author.steamid", "author_steamid"]
TIME_COL = "timestamp_created"
MIN_REVIEW_CHARS = 30
RNG_SEED = PROJECT_RANDOM_SEED
MAX_USERS_EVAL = 5000
rng = np.random.default_rng(RNG_SEED)

_tmp = pd.read_parquet(EVAL_PARQUET, columns=None)
user_col = next((c for c in USER_COL_CANDIDATES if c in _tmp.columns), None)
if user_col is None:
    raise KeyError(
        f"Could not find user id column in {EVAL_PARQUET.name}. Tried: {USER_COL_CANDIDATES}"
    )

usecols = [user_col, "app_id", "review", "recommended", "review_id", TIME_COL]
val_df = pd.read_parquet(EVAL_PARQUET, columns=usecols)

val_df = val_df.loc[val_df["recommended"] == 1].copy()
val_df["review"] = val_df["review"].fillna("").astype(str)
val_df = val_df[val_df["review"].str.len() >= MIN_REVIEW_CHARS]
val_df = val_df[val_df["app_id"].isin(set(app_ids_common.tolist()))]
val_df["ts"] = pd.to_numeric(val_df[TIME_COL], errors="coerce")
val_df = val_df.dropna(subset=[user_col, "app_id", "review", "ts"])
val_df["ts"] = val_df["ts"].astype(np.float64)

print("Eval split:", EVAL_SPLIT_NAME, "|", EVAL_PARQUET.name)
print("User col:", user_col)
print("Positive eval rows on common game universe:", len(val_df))
print("Unique users:", val_df[user_col].nunique())
print("Unique games:", val_df["app_id"].nunique())

Eval split: val | steam_reviews_cleaned_english_val_norm.parquet
User col: author.steamid
Positive eval rows on common game universe: 942340
Unique users: 933929
Unique games: 315


In [18]:
KS_PROXY = (5, 10, 20)

uc = val_df.groupby(user_col)["app_id"].nunique()
multi = uc[uc >= 2].index
multi = pd.Index(rng.permutation(multi.values)[: min(len(multi), int(MAX_USERS_EVAL))])

proxy_rows: list[dict] = []
for uid in multi:
    sub_v = val_df[val_df[user_col] == uid]
    apps = sub_v["app_id"].unique().tolist()

    q_app = int(rng.choice(apps))
    rows_q = sub_v[sub_v["app_id"] == q_app]
    qi = int(rng.integers(0, len(rows_q)))
    row_q = rows_q.iloc[qi]

    query_text = str(row_q["review"])
    positives = {int(a) for a in apps if int(a) != q_app}
    if not positives:
        continue

    proxy_rows.append(
        {
            "steamid": uid,
            "query_text": query_text,
            "anchor_app_id": q_app,
            "positives": positives,
            "n_pos": len(positives),
        }
    )

print(f"Examples: {len(proxy_rows)} users (multi-review {EVAL_SPLIT_NAME}, thumbs-up, in index)")
if proxy_rows:
    print(
        "Positives per example: min",
        min(e["n_pos"] for e in proxy_rows),
        "max",
        max(e["n_pos"] for e in proxy_rows),
    )
proxy_rows[0] if proxy_rows else None

Examples: 5000 users (multi-review val, thumbs-up, in index)
Positives per example: min 1 max 1


{'steamid': 76561198050497106,
 'query_text': "Amazing story line. Amazing everything. I love this game so much and can't wait to play episode 2 ♥♥♥♥",
 'anchor_app_id': 207610,
 'positives': {620},
 'n_pos': 1}

In [ ]:
def hit_rate_at_k(ranked_app_ids: np.ndarray, positives: set[int], k: int) -> float:
    top = set(int(a) for a in ranked_app_ids[:k])
    return 1.0 if (top & positives) else 0.0


def summarize_proxy(agg: dict[str, list[float]]) -> dict[str, float]:
    """Match recs_004 summarization: nanmean for recall/map/ndcg/mrr; mean for hit."""
    out: dict[str, float] = {}
    for metric, vals in agg.items():
        a = np.asarray(vals, dtype=np.float64)
        if metric.startswith("recall") or metric.startswith("map") or metric == "mrr" or metric.startswith("ndcg"):
            out[metric] = float(np.nanmean(a))
        else:
            out[metric] = float(a.mean())
    return out


arms_proxy = {
    "raw_raw": dict(matrix=X_raw_common, structured_query=False),
    "structured_raw": dict(matrix=X_raw_common, structured_query=True),
    "raw_structured": dict(matrix=X_struct_common, structured_query=False),
    "structured_structured": dict(matrix=X_struct_common, structured_query=True),
}

metric_store: dict[str, dict[str, list[float]]] = {}
for arm in arms_proxy:
    metric_store[arm] = {
        **{f"hit@{k}": [] for k in KS_PROXY},
        **{f"recall@{k}": [] for k in KS_PROXY},
        **{f"map@{k}": [] for k in KS_PROXY},
        **{f"ndcg@{k}": [] for k in KS_PROXY},
        "mrr": [],
    }

for row in proxy_rows:
    q = row["query_text"]
    positives = set(row["positives"])
    exclude = {int(row["anchor_app_id"])}

    for arm_name, cfg in arms_proxy.items():
        hits = _top_k_from_matrix(
            retriever,
            q,
            k=max(KS_PROXY),
            matrix=cfg["matrix"],
            app_ids=app_ids_common,
            idx_df=idx_common,
            structured_query=cfg["structured_query"],
            exclude_app_ids=exclude,
        )

        app_ids_ranked = hits["app_id"].to_numpy(dtype=np.int64)

        for k in KS_PROXY:
            metric_store[arm_name][f"hit@{k}"].append(hit_rate_at_k(app_ids_ranked, positives, k))
            metric_store[arm_name][f"recall@{k}"].append(recall_at_k(app_ids_ranked, positives, k))
            metric_store[arm_name][f"map@{k}"].append(average_precision_at_k(app_ids_ranked, positives, k))
            metric_store[arm_name][f"ndcg@{k}"].append(ndcg_at_k(app_ids_ranked, positives, k))
        metric_store[arm_name]["mrr"].append(mrr(app_ids_ranked, positives))

summary_proxy = {arm: summarize_proxy(metrics) for arm, metrics in metric_store.items()}

proxy_table = pd.DataFrame(summary_proxy)

In [20]:
metric_order = [
    "hit@5", "hit@10", "hit@20",
    "recall@5", "recall@10", "recall@20",
    "map@5", "map@10", "map@20",
    "ndcg@5", "ndcg@10", "ndcg@20",
    "mrr",
]

proxy_table = proxy_table.reindex(metric_order)

print(f"Proxy split: {EVAL_SPLIT_NAME} | users: {len(proxy_rows)} | n_games(common): {len(app_ids_common)}")
proxy_table



Proxy split: val | users: 5000 | n_games(common): 315


,raw_raw,structured_raw,raw_structured,structured_structured
hit@5,0.05380,0.02800,0.02180,0.03580
hit@10,0.08060,0.04740,0.03700,0.06520
hit@20,0.13640,0.08260,0.06660,0.11940
recall@5,0.05380,0.02800,0.02180,0.03580
recall@10,0.08060,0.04740,0.03700,0.06520
recall@20,0.13640,0.08260,0.06660,0.11940
map@5,0.02836,0.01515,0.01110,0.01886
map@10,0.03190,0.01771,0.01303,0.02268
map@20,0.03569,0.02004,0.01500,0.02634
ndcg@5,0.03458,0.01828,0.01371,0.02301


In [21]:
PROXY_METRICS_CSV = OUT_DIR / "eval_review_style_4way_proxy_metrics.csv"
proxy_table.to_csv(PROXY_METRICS_CSV)
print("Wrote:", PROXY_METRICS_CSV)

Wrote: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_review_style_4way_proxy_metrics.csv
